In [0]:
# File location and type

file_location = "/FileStore/tables/employees"

file_type = "csv"


# The applied options are for CSV files. For other file types, these will be ignored.

df = spark.read.format(file_type)\
.option("inferSchema", "true")\
.option("header", "true")\
.option("sep",",")\
.load(file_location)
display(df)
 
parquet_output_path = "/dbfs/FileStore/tables/employees_parquet"

df.write.mode("append").parquet(parquet_output_path)
 
# Path to the saved Parquet file

parquet_input_path = "/dbfs/FileStore/tables/employees_parquet"
 
# Read the Parquet file into a DataFrame

df_parquet = spark.read.parquet(parquet_input_path)
 
# Display the DataFrame to verify the content

display(df_parquet)
 

EmployeeID,Name,Department,Salary,JoiningDate
101,John Doe,Engineering,75000,2020-05-15
102,Jane Smith,Marketing,65000,2019-08-20
103,Samuel Green,HR,60000,2021-01-10
104,Emily Johnson,Finance,72000,2018-11-05
105,Michael Brown,Engineering,80000,2022-03-25
101,John Doe,Engineering,75000,2020-05-15
102,Jane Smith,Marketing,65000,2019-08-20
103,Samuel Green,HR,60000,2021-01-10
104,Emily Johnson,Finance,72000,2018-11-05
105,Michael Brown,Engineering,80000,2022-03-25


EmployeeID,Name,Department,Salary,JoiningDate
101,John Doe,Engineering,75000,2020-05-15
102,Jane Smith,Marketing,65000,2019-08-20
103,Samuel Green,HR,60000,2021-01-10
104,Emily Johnson,Finance,72000,2018-11-05
105,Michael Brown,Engineering,80000,2022-03-25
101,John Doe,Engineering,75000,2020-05-15
102,Jane Smith,Marketing,65000,2019-08-20
103,Samuel Green,HR,60000,2021-01-10
104,Emily Johnson,Finance,72000,2018-11-05
105,Michael Brown,Engineering,80000,2022-03-25


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
from pyspark.sql.functions import sum
 
# Step 1: Define a schema for the DataFrame
schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", DoubleType(), True)
])
 
# Step 2: Define some sample data
data = [
    (1, "Alice", "HR", 55000.0),
    (2, "Bob", "Engineering", 80000.0),
    (3, "Charlie", "Marketing", 65000.0),
    (4, "David", "Engineering", 85000.0),
    (5, "Eva", "HR", 60000.0)
]
 
# Step 3: Create the DataFrame using the sample data and schema
df = spark.createDataFrame(data, schema)
display(df)
 
# Add the ranking column
print('EMPLOYEE SALARY RANK :')
df_ranked = df.withColumn("Salary_Rank", dense_rank().over(Window.orderBy("Salary")))
display(df_ranked)
 
# GROUPING SALARY DEPT WISE
print('SUM OF SALARY BY DEPARTMENT :')
df_salary_sum = df.groupBy("Department").agg(sum("Salary").alias("Total_Salary")).orderBy("Total_Salary")
display(df_salary_sum)
 
 
# Step 1: Define the path where you want to save the Delta table
delta_output_path = "/dbfs/FileStore/tables/employees_rank_delta"
 
# Step 2: Save the DataFrame in Delta format
df_ranked.write.format("delta").mode("overwrite").save(delta_output_path)
 
# Optional: Register the Delta table in the metastore for SQL querying
spark.sql("CREATE TABLE IF NOT EXISTS employees_rank_delta USING DELTA LOCATION '/dbfs/FileStore/tables/employees_rank_delta'")
 
 
%sql
#SELECT * FROM employees_rank_delta;

EmployeeID,Name,Department,Salary
1,Alice,HR,55000.0
2,Bob,Engineering,80000.0
3,Charlie,Marketing,65000.0
4,David,Engineering,85000.0
5,Eva,HR,60000.0


EMPLOYEE SALARY RANK :


EmployeeID,Name,Department,Salary,Salary_Rank
1,Alice,HR,55000.0,1
5,Eva,HR,60000.0,2
3,Charlie,Marketing,65000.0,3
2,Bob,Engineering,80000.0,4
4,David,Engineering,85000.0,5


SUM OF SALARY BY DEPARTMENT :


Department,Total_Salary
Marketing,65000.0
HR,115000.0
Engineering,165000.0


UsageError: Line magic function `%sql` not found.
